In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# 대화가능한 인공지능도 만들어보자 논문말고 앞으로 상용화시킬 인공지능도 만들어 두어야한다.

# 1. 피질기둥 클래스

In [ ]:
class CorticalColumn(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_heads: int
    ):
        super().__init__()

        self.num_heads = num_heads

        # 각 헤드마다 흥분(EPSP) / 억제(IPSP) 분리
        self.excitatory = nn.ModuleList([
            nn.Linear(input_dim, hidden_dim)
            for _ in range(num_heads)
        ])

        self.inhibitory = nn.ModuleList([
            nn.Linear(input_dim, hidden_dim)
            for _ in range(num_heads)
        ])

    def forward(self, x):
        """
        x: [batch, seq_len, input_dim]
        """
        head_outputs = []

        for i in range(self.num_heads):
            epsp = F.relu(self.excitatory[i](x))        # EPSP
            ipsp = -F.relu(self.inhibitory[i](x))       # IPSP
            head_outputs.append(epsp + ipsp)

        # [batch, seq_len, num_heads, hidden_dim]
        return torch.stack(head_outputs, dim=2)


# 2-1️⃣ 한글 → 초성 토큰화

In [ ]:
CHOSUNG_LIST = [
    "ㄱ","ㄲ","ㄴ","ㄷ","ㄸ","ㄹ","ㅁ","ㅂ","ㅃ",
    "ㅅ","ㅆ","ㅇ","ㅈ","ㅉ","ㅊ","ㅋ","ㅌ","ㅍ","ㅎ"
]

def tokenize_chosung(sentence):
    tokens = []
    for ch in sentence:
        code = ord(ch) - 0xAC00
        if 0 <= code <= 11171:
            cho = code // (21 * 28)
            tokens.append(cho)
    return tokens


# 2-2️⃣ 초성 → 임베딩 (입력층 = L4)

In [ ]:
class PhonemeInputLayer(nn.Module):
    def __init__(self, num_tokens, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_tokens, embed_dim)

    def forward(self, token_ids):
        return self.embedding(token_ids)


# 3️⃣ 전체 흐름 예제 (말 → 피질기둥 활성)

In [ ]:
# 하이퍼파라미터
NUM_CHOSUNG = 19
EMBED_DIM = 64
HIDDEN_DIM = 128
NUM_HEADS = 6


In [ ]:
# 모듈 생성
input_layer = PhonemeInputLayer(NUM_CHOSUNG, EMBED_DIM)
cortical_column = CorticalColumn(
    input_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_heads=NUM_HEADS
)


In [ ]:
# 문장 입력 (말하기 / 내적 언어)
sentence = "한글은 아름답다"
token_ids = tokenize_chosung(sentence)
token_ids = torch.tensor(token_ids).unsqueeze(0)  # [batch=1, seq]


In [ ]:
# L4 입력
x = input_layer(token_ids)  # [1, seq, embed_dim]

# 피질기둥 처리
out = cortical_column(x)
print(out.shape)


In [ ]:
[1, seq_len, num_heads, hidden_dim]
